<a href="https://colab.research.google.com/github/naceurarij2-design/Multilingual-Summarization-chatbot/blob/main/Model_Evaluation_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install --upgrade torchao
!pip install transformers datasets peft evaluate rouge_score bleu fitz PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [6]:
import os
import torch
import pandas as pd
from datasets import Dataset
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
import evaluate

# =====================================================================
# 1. PIPELINE PATH VERIFICATION
# =====================================================================
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/New_Multilingual_AI_Project"
CORPUS_PATH = f"{PROJECT_DIR}/processed_outputs/Master_Training_Corpus.xlsx"
ADAPTER_PATH = f"{PROJECT_DIR}/trained_model_checkpoints/final_multilingual_adapter"
BASE_MODEL = "facebook/mbart-large-50"

print("📦 Loading Metric Frameworks (ROUGE & BLEU)...")
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

# =====================================================================
# 2. RELOAD DATA & SYNCHRONIZE PROMPT LAYOUTS
# =====================================================================
print("📊 Preparing Test Split Layer...")
df = pd.read_excel(CORPUS_PATH).dropna(subset=['document_body', 'target_summary'])
hf_dataset = Dataset.from_pandas(df)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
# this is absolutly nessacery because :
#If the  text inputs are sent to the GPU (cuda), but my model is still sitting on the computer's basic processor (cpu), PyTorch will instantly crash with a fatal RuntimeError:
#Expected all tensors to be on the same device.
model.eval()

# 🔥 FIXED: Added prompt formatting to match training expectations
def transform_and_tokenize(examples):
    formatted_inputs = []
    for doc in examples["document_body"]:
        # Recreates the exact input prompt style the model learned in Phase 3
        formatted_inputs.append(f"<SUMMARY>: {doc}")

    model_inputs = tokenizer(formatted_inputs, max_length=1024, truncation=True, padding=False)
    return model_inputs

tokenized_dataset = hf_dataset.map(transform_and_tokenize, batched=True)
dataset_splits = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
test_dataset = dataset_splits["test"]

# =====================================================================
# 3. METRIC COMPUTATION ENGINE (SYNCHRONIZED WITH GENERATION)
# =====================================================================
print(f"🚀 Running validation inference across {len(test_dataset)} test samples...")

predictions = []
references = []

for idx, sample in enumerate(test_dataset):
    lang_protocol = sample.get('language_protocol', 'en_XX')
    forced_bos_token_id = tokenizer.lang_code_to_id.get(lang_protocol, tokenizer.lang_code_to_id['en_XX'])

    input_ids = torch.tensor([sample["input_ids"]]).to(device)
    attention_mask = torch.tensor([sample["attention_mask"]]).to(device)

    with torch.no_grad():
        # 🔥 FIXED: Generation parameters matched with Phase 1/2 settings
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            forced_bos_token_id=forced_bos_token_id,
            max_length=250,
            num_beams=6,                  # Matched to your generation standard
            no_repeat_ngram_size=3,       # Prevents repetition artifact skewing
            length_penalty=1.0,           # Prevents bloated lengths from artificially boosting ROUGE
            early_stopping=True
        )

    pred_str = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()
    ref_str = sample["target_summary"].strip()

    predictions.append(pred_str)
    references.append(ref_str)

    if (idx + 1) % 5 == 0 or (idx + 1) == len(test_dataset):
        print(f"   📊 Processed sample [{idx+1}/{len(test_dataset)}]")

print("\n🧮 Compiling Statistical Metrics...")
rouge_results = rouge_metric.compute(predictions=predictions, references=references, use_stemmer=True)
bleu_results = bleu_metric.compute(predictions=predictions, references=references)

# =====================================================================
# 4. FINAL REPORT PRINT LAYER
# =====================================================================
print("\n" + "="*50)
print("🏆 PIPELINE PERFORMANCE BENCHMARK REPORT")
print("="*50)
print(f"🔹 ROUGE-1 (Unigram Overlap):   {rouge_results['rouge1']*100:.2f}%")
print(f"🔹 ROUGE-2 (Bigram Overlap):    {rouge_results['rouge2']*100:.2f}%")
print(f"🔹 ROUGE-L (Sentence Flow):    {rouge_results['rougeL']*100:.2f}%")
print(f"🔹 BLEU Score (Translation Acc): {bleu_results['bleu']*100:.2f}%")
print("="*50)

log_df = pd.DataFrame([{
    "rouge1": rouge_results['rouge1'],
    "rouge2": rouge_results['rouge2'],
    "rougeL": rouge_results['rougeL'],
    "bleu": bleu_results['bleu']
}])
log_df.to_csv(f"{PROJECT_DIR}/processed_outputs/evaluation_metrics.csv", index=False)
print("💾 Validation metrics successfully recorded in your Drive logs directory.")

📦 Loading Metric Frameworks (ROUGE & BLEU)...
📊 Preparing Test Split Layer...


Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/16 [00:00<?, ? examples/s]

🚀 Running validation inference across 2 test samples...
   📊 Processed sample [2/2]

🧮 Compiling Statistical Metrics...

🏆 PIPELINE PERFORMANCE BENCHMARK REPORT
🔹 ROUGE-1 (Unigram Overlap):   9.73%
🔹 ROUGE-2 (Bigram Overlap):    6.74%
🔹 ROUGE-L (Sentence Flow):    9.73%
🔹 BLEU Score (Translation Acc): 0.02%
💾 Validation metrics successfully recorded in your Drive logs directory.
